# DFM — Droplet-Film ordinal model (`dfm_src.DFT`)

**Physics:** predict `Q_cr` from well features, form `log(Q_cr / Q_meas)`, and fit ordinal thresholds **τ₁, τ₂** with per-well **α**.

1. **Classic `Helm`** — ordinal labels 0/1/2.
2. **Dynamic-bounds split** — same CSV + split as TabPFN / QLattice; section **B** saves **`notebook_exports/dfm_*.csv`** and shows **Turner-style plots** using physics `Q_cr` from `predict_qcr`.

Default CSV: `default_qlattice_csv_path()`.

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
from pathlib import Path

from src.utils import Helm, default_qlattice_csv_path
from src.dfm_src import DFT, loading_dynamic_bounds_to_ordinal

data_path = default_qlattice_csv_path()
NOTEBOOK_EXPORT = Path("notebook_exports")
NOTEBOOK_EXPORT.mkdir(parents=True, exist_ok=True)

includ_cols = [
    "Dia", "Dev(deg)", "Area (m2)", "z", "GasDens", "LiquidDens",
    "g (m/s2)", "P/T", "friction_factor", "critical_film_thickness",
]

## A) Classic Helm + hyperparameter search (`evolv_dfm_model`)

Raw physical units — **`scale=False`** for DFT.

In [ ]:
D_classic = Helm(path=data_path, includ_cols=includ_cols, test_size=0.20, scale=False)


def build_dfm(hparams):
    return DFT(
        feature_tol=hparams.get("feature_tol", 1.0),
        dev_tol=hparams.get("dev_tol", 1e-3),
        multiple_dev_policy=hparams.get("multiple_dev_policy", "max"),
    )


hparam_grid = {
    "feature_tol": [0.5, 1.0, 2.0],
    "multiple_dev_policy": ["max", "mean"],
}

trained_dfm = D_classic.evolv_dfm_model(
    build_model=build_dfm,
    hparam_grid=hparam_grid,
    k_folds=5,
)

## B) Same split as QLattice / TabPFN + CSV export + plots

Maps **Loaded=1, Near=0, Unloaded=-1** → DFT ordinal **0,1,2**, then exports **`Q_meas`** and physics **`Q_cr`** with **`Test status`**, and draws Turner + residual figures (same helpers as the QLattice pipeline).

In [ ]:
from src.qlattice_pipeline import (
    eval_turner_train_test_metrics,
    plot_residual_qminus_qmeas,
    plot_turner_qmeas_vs_qrate,
    save_pipeline_predictions_csv,
)

D_db = Helm.from_dynamic_bounds_csv(data_path, seed=42, test_size=0.2)

X_train = np.asarray(D_db.train_X_df.values, dtype=float)
X_test = np.asarray(D_db.test_X_df.values, dtype=float)
load_tr = loading_dynamic_bounds_to_ordinal(D_db.loading_train)
load_te = loading_dynamic_bounds_to_ordinal(D_db.loading_test)

dfm = DFT(feature_tol=1.0, dev_tol=1e-3, multiple_dev_policy="max")
dfm.fit(X_train, D_db.gsflow_train, load_tr)

pred_te = dfm.predict(X_test, D_db.gsflow_test)
print("Test accuracy (dynamic-bounds split):", float(np.mean(pred_te == load_te)))

qcr_tr = dfm.predict_qcr(X_train)
qcr_te = dfm.predict_qcr(X_test)

ev = eval_turner_train_test_metrics(
    qcr_tr,
    D_db.gsflow_train,
    D_db.loading_train,
    qcr_te,
    D_db.gsflow_test,
    D_db.loading_test,
)

q_meas = np.concatenate([D_db.gsflow_train, D_db.gsflow_test])
q_pred = np.concatenate([qcr_tr, qcr_te])
is_train = np.concatenate(
    [np.ones(len(D_db.gsflow_train), dtype=bool), np.zeros(len(D_db.gsflow_test), dtype=bool)]
)
load_dyn = np.concatenate([D_db.loading_train, D_db.loading_test])

save_pipeline_predictions_csv(
    NOTEBOOK_EXPORT,
    pipeline="dfm",
    mode="dynamic_bounds",
    seed=42,
    q_meas=q_meas,
    q_pred=q_pred,
    loading=load_dyn,
    is_train=is_train,
    pred_column="Q_cr_pred_m3day",
    delta_pos=float(ev["dpos"]),
    delta_neg=float(ev["dneg"]),
    source_csv=str(data_path),
)

base = NOTEBOOK_EXPORT / "dfm_dynamic_bounds_turner"
plot_turner_qmeas_vs_qrate(
    q_meas,
    q_pred,
    is_train,
    ev["dpos"],
    ev["dneg"],
    "DFM physics Q_cr vs Q_meas (dynamic-bounds split)",
    r"Physics $Q_{\mathrm{cr}}$ (m³/day)",
    base,
    loading=load_dyn,
    show=True,
)
plot_residual_qminus_qmeas(
    q_meas,
    q_pred,
    is_train,
    ev["dpos"],
    ev["dneg"],
    "DFM: Q_cr - Q_meas (residual)",
    base,
    loading=load_dyn,
    show=True,
)